# AIMNet2: electrostatic head and charge dipoles

Load the default AIMNet2 calculator, inspect its PyTorch model, and expose the Coulomb coupling energy as a separate output. Then calculate a dipole from the charges used by that electrostatic head and compare it with AIMNet2's ASE dipole from partial charges.

Use the demo's `aimnet2` kernel. The first run may download model weights. Run the cells in order from a fresh kernel.

In [ ]:
import os
import sys

# On macOS, avoid mixed-architecture TorchInductor caches. Set before importing torch.
if sys.platform == 'darwin':
    os.environ.setdefault('TORCHDYNAMO_DISABLE', '1')

import copy
import numpy as np
import torch
from ase.build import molecule
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent))
from demo_tools.jhtml import JHTML
from aimnet.calculators import AIMNet2ASE, AIMNet2Calculator

In [ ]:
JHTML.JCard(
    "We will inspect the loaded ", JHTML.Code("calculator.model"),
    " before accessing its output modules. The default model keeps the long range Coulomb term on the calculator and a short range correction inside the torch model.",
    title="AIMNet2 model internals",
    kind="info",
)

In [ ]:
calculator = AIMNet2Calculator()  # package default: aimnet2
model = calculator.model
print(model)

## Expose the electrostatic coupling module

In the default model, the long range Coulomb coupling module belongs to the calculator. The model also has a short range Coulomb correction in its output modules. Copy the calculator's coupling module into `model.outputs`, give that copy a separate output key, and register the key with the calculator. The original total energy calculation remains in place.

In [ ]:
JHTML.JAlert(
    "This notebook uses the default AIMNet2 model and its external Coulomb module. Inspect the printed model before adapting the pattern to another model.",
    kind="info",
)

In [ ]:
print('Model output modules:', list(model.outputs))
coupling_head = calculator.external_coulomb
print('Electrostatic coupling module:', coupling_head)
print('Charge input key:', coupling_head.key_in)
print('Short range correction:', model.outputs['srcoulomb'])

probe = copy.deepcopy(coupling_head)
probe.key_out = 'electrostatic_energy'
model.outputs['electrostatic_probe'] = probe
calculator.keys_out = [*calculator.keys_out, 'electrostatic_energy']

In [ ]:
JHTML.JAlert(
    "AIMNet2 model families can organize charge and Coulomb modules differently. Reusing this exact head path with another model requires inspecting that model first.",
    kind="warning",
)

In [ ]:
JHTML.JAlert(
    "The copied Coulomb module writes a separate electrostatic energy. The original Coulomb and total-energy paths remain active.",
    kind="info",
)

## Calculate energy, charges, and dipoles

The electrostatic module takes the model's `charges` tensor as input. Its dipole from those charges is `sum(q_i r_i)` in e·Å. The ASE wrapper computes its dipole from AIMNet2's partial charges using the same formula and coordinate origin. For this neutral molecule, the two paths should agree.

We will start by optimizing the molecule

In [ ]:
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent))
from demo_tools.optimization import optimize_ase
from ase.optimize import BFGS


In [ ]:
water = molecule('H2O')
water.calc = AIMNet2ASE(base_calc=calculator, charge=0)
optimize_ase(water, optimizer=BFGS, fmax=0.00001)

From here we can compute partial charges and track the electrostatic contributions

In [ ]:
inputs = {'coord': water.positions, 'numbers': water.numbers, 'charge': 0.0}
result = calculator.eval(inputs)

electrostatic_charges = result[coupling_head.key_in].detach().cpu().numpy()
electrostatic_dipole = np.einsum('i,ij->j', electrostatic_charges, water.positions)

partial_charges = water.get_charges()
partial_charge_dipole = water.get_dipole_moment()

print('Total energy (eV):', result['energy'].item())
print('Electrostatic coupling energy (eV):', result['electrostatic_energy'].item())
print('Electrostatic-head charges (e):', electrostatic_charges)
print('AIMNet2 partial charges (e):', partial_charges)
print('Electrostatic-charge dipole (e·Å):', electrostatic_dipole)
print('AIMNet2 partial-charge dipole (e·Å):', partial_charge_dipole)
print('Difference (e·Å):', electrostatic_dipole - partial_charge_dipole)
np.testing.assert_allclose(electrostatic_charges, partial_charges, rtol=1e-6, atol=1e-6)
np.testing.assert_allclose(electrostatic_dipole, partial_charge_dipole, rtol=1e-6, atol=1e-6)

In [ ]:
JHTML.JAlert(
    "The charge and dipole comparisons passed: both calculations use the same AIMNet2 partial charges for neutral water.",
    kind="success",
)

## Plot both dipoles on the molecule

`Molecule` stores coordinates in Bohr, while the AIMNet2 and ASE positions above are in Å. Convert for the molecular geometry; `Molecule.plot` displays it in Å. The arrows use a common visual scale and are offset by 0.1 Å on either side of the molecular center so coincident vectors remain visible. Blue is the electrostatic-head charge dipole; orange is the ASE partial-charge dipole.

In [ ]:
JHTML.JCard(
    "The plotted dipole is multiplied by five so its direction is visible beside the molecule. The numeric dipole printed above remains in e·Å and is not rescaled.",
    title="Reading the dipole view",
    kind="warning",
)

In [ ]:
from McUtils.Data import UnitsData
from Psience.Molecools import Molecule

water_molecule = Molecule(
    water.get_chemical_symbols(),
    water.positions * UnitsData.convert('Angstroms', 'BohrRadius'),
    bonds=[(0, 1), (0, 2)],
    energy_evaluator='aimnet2',
    dipole_evaluator='aimnet2'
)
water_molecule.plot(backend='x3d', dipole=5*partial_charge_dipole)

### Infrared Spectroscopy

In [ ]:
water_molecule.get_normal_modes().freqs * UnitsData.convert("Hartrees", "Wavenumbers")

In [ ]:
water_molecule.get_harmonic_spectrum().plot()